# SFT Training

This notebook trains Qwen3.5-9B-Base on our prepared Tulu 3 mixtures using Tinker. Run from the project root after completing `data_preparation.ipynb`.

## 1. Additional Environment Setup

Create a `.env` file in the project root containing `TINKER_API_KEY=your-api-key-here`. The training script loads it automatically. Do not commit `.env`.

## 2. Train a Model

Start with the balanced 5k dataset, one epoch, and the fixed validation set. `--no_publish` keeps the checkpoint unpublished during experiments. Training uses Tinker credits.

In [ ]:
!python train_sft.py --train_data data/mixtures/smoke_test.jsonl --val_data data/validation.jsonl --num_epochs 1 --batch_size 16 --lr 1e-4 --rank 32 --max_length 2048 --eval_every 100 --seed 42 --checkpoint_name smoke_test --output_dir outputs/smoke_test --no_publish

## 3. View Training Results

The output directory contains `checkpoint_info.json` and the training/validation loss history. Use a different output directory and checkpoint name for each experiment.

In [ ]:
import json
from pathlib import Path

info = json.loads(Path("outputs/balanced_5k/checkpoint_info.json").read_text())
print(json.dumps(info, indent=2))

## 4. Evaluate the Checkpoint

Run the provided evaluation script on the saved sampling checkpoint. Validation loss measures fit to reference responses; benchmark scores measure task performance.

In [ ]:
checkpoint_path = info["checkpoint_path"]
base_model = info["base_model"]

!python -m evaluation.eval_all --checkpoint_path "{checkpoint_path}" --base_model "{base_model}"

## 5. Command-Line Arguments

| Argument | Default / example | Description |
|---|---|---|
| `--train_data` | Required | Training JSONL path. |
| `--val_data` | `data/validation.jsonl` | Fixed validation JSONL path. |
| `--num_epochs` | 1 | Number of training epochs. |
| `--batch_size` | 16 (example) | Number of samples per batch. |
| `--lr` | 1e-4 | Learning rate. |
| `--rank` | 32 | LoRA rank. |
| `--max_length` | 2048 | Maximum sequence length in tokens. |
| `--eval_every` | 100 | Validation interval in optimizer steps. |
| `--seed` | 42 | Training shuffle seed. |
| `--checkpoint_name` | `balanced_5k` (example) | Tinker checkpoint name. |
| `--output_dir` | `outputs/balanced_5k` (example) | Local experiment output directory. |
| `--no_publish` | False | Skip publishing the sampling checkpoint. |

For other mixtures, change `--train_data`, `--checkpoint_name`, and `--output_dir`. Keep the validation set fixed across experiments.

For the exact defaults supported by the implemented script, run the following command.

In [ ]:
!python train_sft.py --help